<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Ensembles

**[Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/)** · Machine Learning: From Problem to Reliable Model · Module 5, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** explain why many models can beat one, build a random forest and gradient boosting, see which problem each one fixes (variance or bias), compare them fairly with logistic regression, and measure what a complex model costs.

| | |
|---|---|
| **Time** | About 60 minutes, plus the module practice. The notebook runs in about two minutes. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Trees and `evaluate()`, from [Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/). Bias and variance, from [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/#bias-and-variance). The experiment log, from [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log). |
| **You do not need** | The local project. The setup below downloads the data, writes `ticket_model.py` and writes your experiment log so far. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M05-L04-ensembles/ensembles-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: ticket_model.py

The next cell writes `ticket_model.py` as you wrote it in [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/#write-ticket_modelpy): the column lists, `load()` and `build_pipeline()`. Run it. You should see `Writing ticket_model.py` (or `Overwriting`).

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 4, lesson 3 (Preprocess with a pipeline).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

It also writes the 10 rows that you added in the earlier lessons (E01 to E10), so this notebook works without them. If you keep your own log in the local project, skip the last line of the cell.

> **Check.** You should see the log with 10 rows, from E01 to E10.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))


# The rows of the earlier lessons (id, change, cv_ap, cv_std, valid_ap, decision, reason).
EARLIER_ROWS = [
    ('E01', 'Baseline: one score for every ticket (the train rate)', 0.117, 0.000, 0.155, 'baseline', 'The floor: any useful model must be above it.'),
    ('E02', "Baseline: the rule 'flag priority 1', as a 0/1 score", 0.178, 0.008, 0.228, 'baseline', 'What Larkfield does today: the model must beat it.'),
    ('E03', 'Logistic regression, the 10 C04 features', 0.327, 0.012, 0.353, 'keep', 'Far above both baselines; train AP 0.328, so it does not overfit.'),
    ('E04', "E03 with class_weight='balanced'", 0.323, 0.012, None, 'reject', 'Lower than E03 on all 5 folds; one more setting for no gain.'),
    ('E05', "Baseline: the rule 'flag priority 1 or team payment'", 0.172, 0.006, None, 'reject', 'Lower than the simpler rule E02.'),
    ('E06', 'Decision tree, max_depth=3', 0.234, 0.011, None, 'reject', 'Far below E03.'),
    ('E07', 'Decision tree, max_depth=6', 0.258, 0.010, None, 'reject', 'The best of the three trees, still far below E03.'),
    ('E08', 'Decision tree, no depth limit', 0.137, 0.004, None, 'reject', 'It memorises train (train AP 1.000) and fails on new tickets.'),
    ('E09', 'Decision tree, no depth limit, min_samples_leaf=100', 0.278, 0.017, None, 'reject', 'The best tree so far, still far below E03.'),
    ('E10', 'E03 with C=0.0001 (a very strong penalty)', 0.302, 0.012, None, 'reject', 'It underfits: both train and CV AP fall.'),
]
for row in EARLIER_ROWS:
    log_experiment(*row)
show_log()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The problem: one tree is unstable

A deep tree has **high variance**: trained on five random halves of train, it gave one May ticket the scores 0, 1, 0, 1 and 0. Limiting the tree lowers the variance but raises the bias. This lesson combines many models into one, to keep the flexibility and lose most of the randomness.

Run the next cell. It loads the data and defines the helpers of the earlier lessons: `cv`, `tree_pipeline()`, `evaluate()` and `table()`.

In [ ]:
import io
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def tree_pipeline(model):
    """build_pipeline() without the scaler: a tree does not need it."""
    return build_pipeline(model).set_params(prepare__numbers__scale="passthrough")


def evaluate(pipeline):
    """Train AP (fit on all train tickets) and CV AP (mean ± std of the 5 folds)."""
    fold_scores = cross_val_score(pipeline, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
    pipeline.fit(train[FEATURES], train[TARGET])
    train_ap = average_precision_score(train[TARGET], pipeline.predict_proba(train[FEATURES])[:, 1])
    return {"train AP": round(float(train_ap), 3), "CV AP": round(float(fold_scores.mean()), 3),
            "std": round(float(fold_scores.std()), 3)}


def table(results):
    """The results as a table, one row per model (rows still set to ... are left out)."""
    return pd.DataFrame({name: r for name, r in results.items() if isinstance(r, dict)}).T


print(len(train), "train tickets,", len(valid), "valid tickets")

> **Check.** You should see `16939 train tickets, 1179 valid tickets`.

## 2. The wisdom of the crowd

Ask 100 people to guess the number of sweets in a jar. Single guesses are far off, but their average is often close, because each person's error is their own, and errors that go in different directions cancel. If everyone reads the same wrong hint, their errors are the same, and the average keeps the error.

The next cell simulates both crowds for a jar with 437 sweets.

> **Predict.** In which crowd is the average guess closer to 437? Run the cell.

In [ ]:
rng = np.random.default_rng(0)
true_count = 437
own_errors = np.round(true_count + rng.normal(0, 120, 100))      # each person makes their own error
same_hint = np.round(true_count + 150 + rng.normal(0, 40, 100))  # everyone read the same wrong hint
for name, guesses in [("own errors", own_errors), ("same hint", same_hint)]:
    print(f"{name:10s} one guess misses by {np.abs(guesses - true_count).mean():.0f} on average; "
          f"the average guess is {guesses.mean():.0f}, a miss of {abs(guesses.mean() - true_count):.0f}")

> **Check.** Own errors: one guess misses by `95`, the average is `447`, a miss of `10`. Same hint: one guess misses by `148`, the average is `585`, a miss of `148`.

Run the next cell to draw the two crowds.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, (name, guesses) in zip(axes, [("own errors", own_errors), ("same hint", same_hint)]):
    ax.hist(guesses, bins=range(100, 901, 50), color="C0")
    ax.axvline(true_count, color="black", linestyle="--", label=f"true {true_count}")
    ax.axvline(guesses.mean(), color="C1", linestyle="--", label=f"average {guesses.mean():.0f}")
    ax.set_title(name)
    ax.set_xlabel("guessed number of sweets")
    ax.legend()
axes[0].set_ylabel("people")
plt.tight_layout()
plt.show()

> **Check.** On the left, the guesses spread widely around the true count, and the average line is close to it. On the right, the guesses are close together, all too high.

A model made of many models is an **ensemble**. It helps when two conditions are true: **each member is useful** on its own, and **the members make different errors**.

## 3. Bagging and the random forest

A **random forest** trains many trees independently. Each tree learns from its own **bootstrap sample**: as many rows as train, drawn at random *with replacement*, so some rows come twice and some not at all. At each split, a tree chooses from a random part of the features. The forest **averages** the trees' scores. (Bagging, short for *bootstrap aggregating*, is the same without the random features.)

Run the cell: a forest of 5 small trees on the 10 tickets of *Decision trees*, and a new ticket with priority 1 and 100 words.

In [ ]:
tiny = pd.DataFrame({
    "ticket": [f"T-{i:02d}" for i in range(1, 11)],
    "priority": [1, 1, 1, 1, 2, 2, 2, 3, 3, 3],
    "word_count": [150, 40, 120, 45, 130, 160, 20, 30, 145, 60],
    "escalated": [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
})

tiny_forest = RandomForestClassifier(n_estimators=5, max_depth=2, random_state=0)
tiny_forest.fit(tiny[["priority", "word_count"]], tiny["escalated"])
print("tree 1 learned from:", sorted(tiny["ticket"].iloc[tiny_forest.estimators_samples_[0]]))

new_ticket = pd.DataFrame({"priority": [1], "word_count": [100]})
each_tree = [round(float(tree.predict_proba(new_ticket.to_numpy())[0, 1]), 2) for tree in tiny_forest.estimators_]
print("score of each tree:", each_tree)
print("forest score (the average):", round(float(tiny_forest.predict_proba(new_ticket)[0, 1]), 2))

> **Check.** Tree 1 learned from T-01, T-03, T-04 and T-10 twice each, and T-07 and T-08 once. The scores of the trees are `[1.0, 0.4, 1.0, 1.0, 1.0]`, and the forest score is `0.88`.

Does the randomness matter? The next cell averages 1, 10 and 50 deep trees in two ways: **similar trees** (every tree sees all the rows and all the features) and a **random forest**. It takes about 30 seconds.

> **Predict.** Which kind improves more when you average more trees?

In [ ]:
for kind, options in [("similar trees", dict(bootstrap=False, max_features=None)),
                      ("random forest", dict(bootstrap=True, max_features="sqrt"))]:
    for n_trees in (1, 10, 50):
        forest = RandomForestClassifier(n_estimators=n_trees, n_jobs=-1, random_state=0, **options)
        folds = cross_val_score(tree_pipeline(forest), train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
        print(f"{kind:14s} {n_trees:3d} deep trees: CV AP {folds.mean():.3f}")

> **Check.** Similar trees: `0.139`, `0.148`, `0.149`. Random forest: `0.137`, `0.220`, `0.272`.

Similar trees make the same errors, so their average hardly improves. The forest's trees are different, so their errors cancel. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#why-it-helps-averaging-lowers-the-variance) has the chart up to 200 trees.

Now repeat the five-halves test of *Control complexity* with a forest of 200 deep trees. Run the cell. It takes about 10 seconds.

In [ ]:
ticket = valid[valid["ticket_id"] == "T-117529"]
for name, make in [("one tree, no limit", lambda: tree_pipeline(DecisionTreeClassifier(random_state=0))),
                   ("forest, 200 trees", lambda: tree_pipeline(
                       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)))]:
    ticket_scores, all_scores = [], []
    for seed in range(5):
        half = train.sample(frac=0.5, random_state=seed)
        pipeline = make().fit(half[FEATURES], half[TARGET])
        ticket_scores.append(round(float(pipeline.predict_proba(ticket[FEATURES])[0, 1]), 2))
        all_scores.append(pipeline.predict_proba(valid[FEATURES])[:, 1])
    spread = pd.DataFrame(all_scores).std(ddof=0).mean()
    print(f"{name:20s} T-117529 gets {ticket_scores}; mean spread over May {spread:.3f}")

> **Check.** One tree: T-117529 gets `[0.0, 1.0, 0.0, 1.0, 0.0]`, mean spread `0.194`. Forest: a mean spread of `0.034`.

**Averaging lowers the variance**: the forest's answers move about six times less when the train tickets change.

## 4. Boosting: learn from the mistakes

**Gradient boosting** builds small trees one after another, like a student who studies only the questions she got wrong last time. Every ticket starts with the escalation rate. Each new tree learns the errors that are left, and adds a part of its correction: the **learning rate** sets the part.

Run the cell: two rounds by hand on the 10 tickets, with a learning rate of 0.5.

In [ ]:
x_tiny = tiny[["priority", "word_count"]]
boost = tiny[["ticket", "escalated"]].copy()
boost["start"] = 0.4                                # the escalation rate, for every ticket
score = boost["start"]
for step in (1, 2):
    error = tiny["escalated"] - score               # what the scores still miss
    small_tree = DecisionTreeRegressor(max_depth=1).fit(x_tiny, error)
    print(f"tree {step} asks:", export_text(small_tree, feature_names=["priority", "word_count"]).split()[1:4])
    score = score + 0.5 * small_tree.predict(x_tiny)  # learning rate 0.5: half of the correction
    boost[f"after tree {step}"] = score.round(3)
print(boost.to_string(index=False))

> **Check.** Tree 1 asks about `priority <= 1.50`; tree 2 about `word_count <= 42.50`. After tree 2, T-01, T-03 and T-04 have 0.657, and T-02 has gone down to 0.385.

Boosting attacks the other problem: its members are small, so each one alone has **high bias**. The next cell boosts trees with one question each (`max_depth=1`), and scores the model after 1, 20 and 200 trees. It takes about 10 seconds.

> **Predict.** One tree with one question is the rule "flag priority 1" (0.178). Where will 200 such trees end?

In [ ]:
for n_trees in (1, 20, 200):
    stumps = HistGradientBoostingClassifier(max_depth=1, learning_rate=0.1, max_iter=n_trees,
                                            early_stopping=False, random_state=0)
    print(f"{n_trees:3d} trees:", evaluate(tree_pipeline(stumps)))

> **Check.** 1 tree: `0.178` and `0.178`. 20 trees: CV AP `0.279`. 200 trees: train AP `0.341`, CV AP `0.333`.

**Adding corrections lowers the bias**: simple members, a strong sum. Boosting can also overfit, with many deep trees or a large learning rate, so it has more hyperparameters: the number of trees, the learning rate, the depth.

## 5. Worked example: the ensembles on Larkfield's tickets

Score three ensembles next to logistic regression, on the same folds.

> **Predict.** Will any ensemble beat logistic regression's 0.327 ± 0.012 by more than one standard deviation? Run the cell. It takes about 20 seconds.

In [ ]:
results = {
    "logistic regression": evaluate(build_pipeline()),
    "forest, default": evaluate(tree_pipeline(
        RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0))),
    "forest, min leaf 20": evaluate(tree_pipeline(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0))),
    "boosting, default": evaluate(tree_pipeline(HistGradientBoostingClassifier(random_state=0))),
}
print(table(results))

> **Check.** You should see these rows:
>
> | | train AP | CV AP | std |
> |---|---|---|---|
> | logistic regression | 0.328 | 0.327 | 0.012 |
> | forest, default | 1.000 | 0.290 | 0.014 |
> | forest, min leaf 20 | 0.483 | 0.325 | 0.018 |
> | boosting, default | 0.477 | 0.316 | 0.010 |

The best ensembles only **tie** logistic regression: the differences are smaller than one standard deviation. The pattern at Larkfield is mostly additive (each fact adds some risk on its own), which is the shape of logistic regression.

Confirm the candidates once on May. Run the cell.

In [ ]:
for name, pipeline in [("logistic regression", build_pipeline()),
                       ("forest, min leaf 20", tree_pipeline(RandomForestClassifier(
                           n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0))),
                       ("boosting, default", tree_pipeline(HistGradientBoostingClassifier(random_state=0)))]:
    pipeline.fit(train[FEATURES], train[TARGET])
    print(f"{name:20s} valid AP {average_precision_score(valid[TARGET], pipeline.predict_proba(valid[FEATURES])[:, 1]):.3f}")

> **Check.** You should see `0.353`, `0.382` and `0.369`.

The forest's lead on May, 0.029, is inside May's chance range of about 0.03 (May has only 183 escalations). Cross-validation said "tie", and May does not contradict it.

## 6. Guided practice: log the ensembles

> **Your turn.** Write the rows E11 (forest, default; no valid AP), E12 (forest, min leaf 20; valid AP 0.382) and E13 (boosting, default; valid AP 0.369) with `log_experiment()`. Use the decision `"reject"` and a reason for each. Run the cell, then the check cell.

In [ ]:
# Your turn: write the rows E11, E12 and E13 (decision "reject"; valid AP only for E12 and E13).
...
show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision"))

In [ ]:
log = pd.read_csv(LOG_FILE).set_index("id")
missing = [row_id for row_id in ['E11', 'E12', 'E13'] if row_id not in log.index]
if missing:
    print("Not yet: the log is missing", ", ".join(missing) + ". Write the rows with log_experiment(), then run this cell again.")
else:
    rows = log.loc[['E11', 'E12', 'E13']]
    expect_hash("the log from E11 to E13", [[r.cv_ap, r.cv_std, None if pd.isna(r.valid_ap) else r.valid_ap, r.decision]
                                   for r in rows.itertuples()], "405984f7a00c978f")

> **Check.** You should see `Check passed: the log from E11 to E13 is right.`

## 7. Your turn: a slower, smaller boosting

Change one condition: `max_depth=3` and `learning_rate=0.05`.

> **Predict.** Will the train AP go up or down from 0.477? Will the CV AP change much?

> **Your turn.** Replace `...` with `evaluate(tree_pipeline(HistGradientBoostingClassifier(...)))`. Run the cell, then the check cell.

In [ ]:
# Your turn: boosting with max_depth=3 and learning_rate=0.05 (and random_state=0).
slow_boosting = ...
print(slow_boosting)

In [ ]:
if not isinstance(slow_boosting, dict):
    print("Not yet: replace ... with evaluate(tree_pipeline(HistGradientBoostingClassifier(...))).")
else:
    expect_hash("your result", slow_boosting, "47f7a8a81fb404eb")

> **Your turn.** Write the row E14. Its valid AP is 0.374. Decide, and write your reason. Run the cell, then the check cell.

In [ ]:
# Your turn: write the row E14 (its valid AP is 0.374).
...
show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision"))

In [ ]:
log = pd.read_csv(LOG_FILE).set_index("id")
missing = [row_id for row_id in ['E14'] if row_id not in log.index]
if missing:
    print("Not yet: the log is missing", ", ".join(missing) + ". Write the rows with log_experiment(), then run this cell again.")
else:
    rows = log.loc[['E14']]
    expect_hash("the row E14", [[r.cv_ap, r.cv_std, None if pd.isna(r.valid_ap) else r.valid_ap, r.decision]
                                   for r in rows.itertuples()], "93a3fe068c959885")

Compare your decision with the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#your-turn-a-slower-smaller-boosting).

## 8. The cost of complexity

A tie in the score is not a tie in the cost. The next cell measures the time to train, the time to score the May tickets, and the size of the saved file, for three models. Run it.

In [ ]:
def costs(name, pipeline):
    """Time to train, time to score the 1,179 May tickets, and the size of the saved file."""
    start = time.perf_counter()
    pipeline.fit(train[FEATURES], train[TARGET])
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    pipeline.predict_proba(valid[FEATURES])
    score_ms = (time.perf_counter() - start) * 1000
    file = io.BytesIO()          # a file in memory
    joblib.dump(pipeline, file)  # save the pipeline as a file, as Module 8 does
    print(f"{name:20s} train {fit_seconds:5.2f} s   score May {score_ms:5.1f} ms   file {file.tell() / 1000:8,.0f} KB")
    return pipeline


simple = costs("logistic regression", build_pipeline())
forest = costs("forest, min leaf 20", tree_pipeline(
    RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0)))
boosted = costs("boosting, default", tree_pipeline(HistGradientBoostingClassifier(random_state=0)))
print("logistic regression:", simple.named_steps["model"].coef_.size, "weights")
print("forest:", sum(tree.get_n_leaves() for tree in forest.named_steps["model"].estimators_), "leaves in 200 trees")

> **Check.** Your times differ from the lesson page, but the files do not: about `5` KB for logistic regression, about `10,459` KB for the forest and about `145` KB for boosting. Then `27 weights` and `64934 leaves in 200 trees`.

The forest is about 2,000 times bigger as a file, and slower. It is also much harder to explain (64,934 leaves against 27 weights), to check, and to debug, and it has more hyperparameters to choose. A complex model must **earn its place** with a gain larger than the noise. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#the-cost-of-complexity) discusses each cost.

## 9. Failure case: the perfect score

Tomás reports the forest with default settings by its score on train. Run the cell.

In [ ]:
print("random forest, default settings:", results["forest, default"])

> **Check.** You should see `train AP 1.0`, `CV AP 0.29` and `std 0.014`.

The default trees grow until every leaf is pure, so each tree memorises its bootstrap sample. **Never judge a model by its score on the tickets it learned from.** The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#the-perfect-score) has the full failure case.

## 10. Module practice: compare two model families

Compare two families under the same validation procedure: the same features, the same folds, the same score; decide with CV, then confirm once on May. `compare()` prints the AP of both models on each fold, and the difference per fold. Run the worked example.

In [ ]:
def compare(name_a, pipeline_a, name_b, pipeline_b):
    """Two pipelines on the same 5 folds of train: AP per fold, mean ± std, the
    difference per fold, and the train AP (fit and scored on all train tickets)."""
    a = cross_val_score(pipeline_a, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
    b = cross_val_score(pipeline_b, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
    for name, pipeline, folds in ((name_a, pipeline_a, a), (name_b, pipeline_b, b)):
        pipeline.fit(train[FEATURES], train[TARGET])
        train_ap = average_precision_score(train[TARGET], pipeline.predict_proba(train[FEATURES])[:, 1])
        print(f"{name:20s} folds {folds.round(3)}  CV AP {folds.mean():.3f} ± {folds.std():.3f}  train AP {train_ap:.3f}")
    print(f"{'b minus a':20s} folds {(b - a).round(3)}  mean {(b - a).mean():+.3f}")
    return a, b


folds_a, folds_b = compare("logistic regression", build_pipeline(), "forest, min leaf 20", tree_pipeline(
    RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0)))

> **Check.** The last line shows `[ 0.002 -0.016 -0.016  0.005  0.015]` and a mean of `-0.002`.

The sign of the difference changes from fold to fold: a tie.

> **Your turn.** Choose two models from two different families, not the pair above. Predict which one wins, run `compare()`, decide from the CV results only, and write one paragraph: the decision, the numbers, the reason. Then confirm once on May. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#module-practice) has a model answer.

In [ ]:
# Your task: two models from two different families (not the pair above).
# For example: compare("logistic regression", build_pipeline(), "my model", tree_pipeline(...))
...

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this module you went around the loop: your log has 14 rows. A random forest averages different deep trees and lowers the variance; boosting adds small corrections and lowers the bias. On Larkfield's tickets, the best ensembles only tie logistic regression, and they cost more. E03, logistic regression, stays. Next, Module 6 improves the model by changing its features.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Select useful features](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m05/ensembles/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.